# Advanced GPU Computing with CUDA Python


In [1]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!pip install -q numba cupy-cuda12x pytest pynvml

import math, time, warnings
import numpy as np
import cupy as cp
import scipy.ndimage, scipy.special, scipy.fft, scipy.sparse
from numba import cuda, float32, int32, int64, uint32
from numba.core.errors import NumbaPerformanceWarning
warnings.simplefilter("ignore", NumbaPerformanceWarning)

if not cuda.is_available():
    raise SystemError("CUDA GPU не найден! Runtime > Change runtime type > T4 GPU")

gpu = cuda.get_current_device()
print("Device:", gpu.name.decode() if isinstance(gpu.name, bytes) else gpu.name)
print("Compute capability:", gpu.compute_capability)
print("Warp size:", gpu.WARP_SIZE, "| Max threads/block:", gpu.MAX_THREADS_PER_BLOCK)
print("Max shared/block: %.1f KB" % (gpu.MAX_SHARED_MEMORY_PER_BLOCK / 1024))

def cdiv(a, b):
    return (a + b - 1) // b

rng = np.random.default_rng(0)

name, driver_version, memory.total [MiB]
Tesla T4, 580.82.07, 15360 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 MB 7.2 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/cupy/_environment.py:677: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda12x, cupy-cuda13x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


Device: Tesla T4
Compute capability: ComputeCapability(major=7, minor=5)
Warp size: 32 | Max threads/block: 1024
Max shared/block: 48.0 KB


# Module 1 — CUDA Execution Model (Tasks 1–8)

In [2]:
# ===== Task 01: vector_add_1d =====
@cuda.jit
def vector_add_1d(a, b, out):
    i = cuda.grid(1)
    if i < out.size:
        out[i] = a[i] + b[i]

for N in [1, 255, 256, 1024, 1000003]:
    a = rng.random(N, dtype=np.float32)
    b = rng.random(N, dtype=np.float32)
    out = np.zeros(N, np.float32)
    vector_add_1d[cdiv(N, 256), 256](a, b, out)
    np.testing.assert_allclose(out, a + b, atol=1e-5)
print("✅ Task 01 passed")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


✅ Task 01 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [3]:
# ===== Task 02: vector_scale_grid_stride =====
@cuda.jit
def vector_scale_grid_stride(x, alpha, out):
    start = cuda.grid(1)
    stride = cuda.gridsize(1)          # gridDim.x * blockDim.x
    for i in range(start, x.size, stride):
        out[i] = x[i] * alpha

N = 10_000_000
x = rng.random(N, dtype=np.float32)
alpha = np.float32(2.5)
d_x = cuda.to_device(x)
d_out = cuda.device_array(N, np.float32)
vector_scale_grid_stride[64, 256](d_x, alpha, d_out)     # 16384 потоков << N
np.testing.assert_allclose(d_out.copy_to_host(), x * alpha, atol=1e-5)
print("✅ Task 02 passed")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


✅ Task 02 passed


In [4]:
# ===== Task 03: matrix_add_2d =====
@cuda.jit
def matrix_add_2d(A, B, C):
    col, row = cuda.grid(2)            # x -> col, y -> row
    M, N = A.shape
    if row < M and col < N:
        C[row, col] = A[row, col] + B[row, col]

M, N = 513, 1021
A = rng.random((M, N), dtype=np.float32)
B = rng.random((M, N), dtype=np.float32)
C = np.zeros((M, N), np.float32)
matrix_add_2d[(cdiv(N, 16), cdiv(M, 16)), (16, 16)](A, B, C)
np.testing.assert_allclose(C, A + B, atol=1e-5)
print("✅ Task 03 passed")

✅ Task 03 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [5]:
# ===== Task 04: matrix_transpose_naive =====
@cuda.jit
def matrix_transpose_naive(A, B):
    col, row = cuda.grid(2)
    M, N = A.shape
    if row < M and col < N:
        B[col, row] = A[row, col]

for (M, N) in [(513, 1021), (64, 64), (1000, 37)]:
    A = rng.random((M, N), dtype=np.float32)
    B = np.zeros((N, M), np.float32)
    matrix_transpose_naive[(cdiv(N, 16), cdiv(M, 16)), (16, 16)](A, B)
    np.testing.assert_allclose(B, A.T, atol=1e-5)
print("✅ Task 04 passed")

✅ Task 04 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [6]:
# ===== Task 05: volume_init_3d =====
@cuda.jit
def volume_init_3d(vol, D, H, W):
    x, y, z = cuda.grid(3)
    if x < W and y < H and z < D:
        flat = z * H * W + y * W + x            # flattened index (для демонстрации)
        vol[z, y, x] = z * 10000 + y * 100 + x

D, H, W = 33, 47, 65
vol = np.zeros((D, H, W), np.int32)
volume_init_3d[(cdiv(W, 8), cdiv(H, 8), cdiv(D, 8)), (8, 8, 8)](vol, D, H, W)
z, y, x = np.meshgrid(np.arange(D), np.arange(H), np.arange(W), indexing="ij")
np.testing.assert_equal(vol, (z * 10000 + y * 100 + x).astype(np.int32))
print("✅ Task 05 passed")

✅ Task 05 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [7]:
# ===== Task 06: rgb_to_grayscale =====
@cuda.jit
def rgb_to_grayscale(img_rgb, img_gray):
    x, y = cuda.grid(2)
    H, W = img_gray.shape
    if y < H and x < W:
        r = img_rgb[y, x, 0]
        g = img_rgb[y, x, 1]
        b = img_rgb[y, x, 2]
        img_gray[y, x] = 0.299 * r + 0.587 * g + 0.114 * b

H, W = 480, 641
# float32
img = rng.random((H, W, 3), dtype=np.float32)
gray = np.zeros((H, W), np.float32)
rgb_to_grayscale[(cdiv(W, 16), cdiv(H, 16)), (16, 16)](img, gray)
np.testing.assert_allclose(gray, 0.299*img[..., 0] + 0.587*img[..., 1] + 0.114*img[..., 2], atol=1e-4)
# uint8
img8 = rng.integers(0, 256, (H, W, 3), dtype=np.uint8)
gray8 = np.zeros((H, W), np.float32)
rgb_to_grayscale[(cdiv(W, 16), cdiv(H, 16)), (16, 16)](img8, gray8)
ref8 = 0.299*img8[..., 0] + 0.587*img8[..., 1] + 0.114*img8[..., 2]
np.testing.assert_allclose(gray8, ref8, atol=1e-3)   # float32 на значениях ~255
print("✅ Task 06 passed")

✅ Task 06 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [8]:
# ===== Task 07: vector_lerp =====
@cuda.jit
def vector_lerp(a, b, t, out):
    i = cuda.grid(1)
    if i < out.size:
        out[i] = (1.0 - t[i]) * a[i] + t[i] * b[i]

N = 1_000_001
a = rng.random(N, dtype=np.float32); b = rng.random(N, dtype=np.float32)
t = rng.random(N, dtype=np.float32); out = np.zeros(N, np.float32)
vector_lerp[cdiv(N, 256), 256](a, b, t, out)
np.testing.assert_allclose(out, (1 - t) * a + t * b, atol=1e-5)
print("✅ Task 07 passed")

✅ Task 07 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [9]:
# ===== Task 08: vector_threshold =====
@cuda.jit
def vector_threshold(x, thresh, mask):
    start = cuda.grid(1)
    stride = cuda.gridsize(1)
    for i in range(start, x.size, stride):
        mask[i] = 1 if x[i] >= thresh else 0     # predication, без ветвления по данным

N = 2_000_003
x = rng.standard_normal(N).astype(np.float32)
thresh = np.float32(0.25)
mask = np.zeros(N, np.uint8)
vector_threshold[128, 256](x, thresh, mask)
np.testing.assert_equal(mask, (x >= thresh).astype(np.uint8))
print("✅ Task 08 passed")

✅ Task 08 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


# Module 2 — Memory Hierarchy (Tasks 9–17)

In [10]:
# ===== Task 09: benchmark_strided_access =====
# stride=1 -> коалесцированное чтение; stride=S -> соседние потоки читают адреса через S элементов.
# Индексация — перестановка, поэтому каждый элемент читается ровно один раз.
@cuda.jit
def benchmark_strided_access(in_arr, out_arr, stride):
    i = cuda.grid(1)
    n = in_arr.size
    if i < n:
        m = n // stride
        idx = (i % m) * stride + i // m
        out_arr[i] = in_arr[idx]

N = 2**24
h_in = rng.random(N, dtype=np.float32)
d_in = cuda.to_device(h_in)
d_out = cuda.device_array(N, np.float32)
grid = cdiv(N, 256)

def bandwidth(stride, reps=20):
    benchmark_strided_access[grid, 256](d_in, d_out, stride)
    cuda.synchronize()
    s, e = cuda.event(), cuda.event()
    s.record()
    for _ in range(reps):
        benchmark_strided_access[grid, 256](d_in, d_out, stride)
    e.record(); e.synchronize()
    ms = cuda.event_elapsed_time(s, e) / reps
    return 2 * N * 4 / ms / 1e6      # GB/s (чтение + запись)

bws = {}
for st in [1, 2, 4, 8, 16, 32]:
    bws[st] = bandwidth(st)
    m = N // st
    i = np.arange(N)
    np.testing.assert_allclose(d_out.copy_to_host(), h_in[(i % m) * st + i // m])
    print(f"stride {st:2d}: {bws[st]:7.1f} GB/s")
assert bws[1] > bws[32], "stride=1 должен быть быстрее stride=32"
print("✅ Task 09 passed")

stride  1:   101.7 GB/s
stride  2:   100.2 GB/s
stride  4:    97.1 GB/s
stride  8:    59.6 GB/s
stride 16:    30.6 GB/s
stride 32:    27.5 GB/s
✅ Task 09 passed


In [12]:
# ===== Task 10: stencil_1d_shared =====
T10_BLOCK = 256
T10_MAXR = 16            # максимальный radius, под который выделен shared

@cuda.jit
def stencil_1d_shared(in_arr, out_arr, radius):
    tile = cuda.shared.array(shape=288, dtype=float32)
    tx = cuda.threadIdx.x
    n = in_arr.size
    base = cuda.blockIdx.x * T10_BLOCK
    k = tx
    while k < T10_BLOCK + 2 * radius:           # центр + halo с зажимом границ (clamp)
        g = base - radius + k
        if g < 0:
            g = 0
        if g > n - 1:
            g = n - 1
        tile[k] = in_arr[g]
        k += T10_BLOCK
    cuda.syncthreads()
    i = base + tx
    if i < n:
        s = float32(0.0)
        for d in range(-radius, radius + 1):
            s += tile[tx + radius + d]
        out_arr[i] = s / (2 * radius + 1)

N = 100_003
for radius in [1, 5, 16]:
    x = rng.random(N, dtype=np.float32)
    out = np.zeros(N, np.float32)
    stencil_1d_shared[cdiv(N, T10_BLOCK), T10_BLOCK](x, out, radius)
    np.testing.assert_allclose(out, scipy.ndimage.uniform_filter1d(x, size=2*radius+1, mode="nearest"), atol=1e-4)
print("✅ Task 10 passed")

✅ Task 10 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [13]:
# ===== Task 11: conv_1d_shared_tiled =====
T11_BLOCK = 256
T11_MAXK = 64

@cuda.jit
def conv_1d_shared_tiled(x, kernel, y):
    tile = cuda.shared.array(320, float32)
    sk = cuda.shared.array(64, float32)
    tx = cuda.threadIdx.x
    n = x.size
    K = kernel.size
    half = (K - 1) // 2              # так же, как np.convolve(mode='same')
    left = K - 1 - half
    base = cuda.blockIdx.x * T11_BLOCK
    if tx < K:
        sk[tx] = kernel[tx]
    t = tx
    while t < T11_BLOCK + K - 1:
        g = base - left + t
        if g >= 0 and g < n:
            tile[t] = x[g]
        else:
            tile[t] = 0.0
        t += T11_BLOCK
    cuda.syncthreads()
    i = base + tx
    if i < n:
        acc = float32(0.0)
        for j in range(K):
            acc += sk[j] * tile[tx + K - 1 - j]
        y[i] = acc

N = 100_000
for K in [3, 15, 31, 64]:
    x = rng.random(N, dtype=np.float32)
    kern = rng.random(K, dtype=np.float32)
    y = np.zeros(N, np.float32)
    conv_1d_shared_tiled[cdiv(N, T11_BLOCK), T11_BLOCK](x, kern, y)
    np.testing.assert_allclose(y, np.convolve(x, kern, mode="same"), rtol=1e-4, atol=1e-4)
print("✅ Task 11 passed")

✅ Task 11 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [15]:
# ===== Task 12: matmul_tiled_shared =====
@cuda.jit
def matmul_tiled_shared(A, B, C):
    sA = cuda.shared.array((16, 16), dtype=float32)
    sB = cuda.shared.array((16, 16), dtype=float32)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    row = cuda.blockIdx.y * 16 + ty
    col = cuda.blockIdx.x * 16 + tx
    M, K = A.shape
    N = B.shape[1]
    acc = float32(0.0)
    for p in range((K + 15) // 16):
        a_col = p * 16 + tx
        b_row = p * 16 + ty
        if row < M and a_col < K:
            sA[ty, tx] = A[row, a_col]
        else:
            sA[ty, tx] = 0.0
        if b_row < K and col < N:
            sB[ty, tx] = B[b_row, col]
        else:
            sB[ty, tx] = 0.0
        cuda.syncthreads()
        for k in range(16):
            acc += sA[ty, k] * sB[k, tx]
        cuda.syncthreads()
    if row < M and col < N:
        C[row, col] = acc

for n in [64, 500, 1024]:
    A = rng.random((n, n), dtype=np.float32)
    B = rng.random((n, n), dtype=np.float32)
    C = np.zeros((n, n), np.float32)
    matmul_tiled_shared[(cdiv(n, 16), cdiv(n, 16)), (16, 16)](A, B, C)
    np.testing.assert_allclose(C, A @ B, rtol=1e-4, atol=1e-3)
print("✅ Task 12 passed")

✅ Task 12 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [16]:
# ===== Task 13: transpose_bank_conflict_free =====
@cuda.jit
def transpose_bank_conflict_free(A, B):
    tile = cuda.shared.array((32, 33), dtype=float32)     # +1 колонка убирает bank conflicts
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    M, N = A.shape
    x = cuda.blockIdx.x * 32 + tx
    y = cuda.blockIdx.y * 32 + ty
    if y < M and x < N:
        tile[ty, tx] = A[y, x]               # коалесцированное чтение
    cuda.syncthreads()
    x2 = cuda.blockIdx.y * 32 + tx           # колонка в B (B: N x M)
    y2 = cuda.blockIdx.x * 32 + ty           # строка в B
    if y2 < N and x2 < M:
        B[y2, x2] = tile[tx, ty]             # коалесцированная запись

for (M, N) in [(1024, 1024), (513, 1021), (100, 33)]:
    A = rng.random((M, N), dtype=np.float32)
    B = np.zeros((N, M), np.float32)
    transpose_bank_conflict_free[(cdiv(N, 32), cdiv(M, 32)), (32, 32)](A, B)
    np.testing.assert_allclose(B, A.T, atol=1e-5)
print("✅ Task 13 passed")

✅ Task 13 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 8 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [17]:
# ===== Task 14: reduction_sum_shared =====
@cuda.jit
def reduction_sum_shared(in_arr, block_sums):
    sdata = cuda.shared.array(256, dtype=float32)
    tx = cuda.threadIdx.x
    i = cuda.grid(1)
    if i < in_arr.size:
        sdata[tx] = in_arr[i]
    else:
        sdata[tx] = 0.0
    cuda.syncthreads()
    stride = cuda.blockDim.x // 2
    while stride > 0:                         # sequential addressing
        if tx < stride:
            sdata[tx] += sdata[tx + stride]
        cuda.syncthreads()
        stride >>= 1
    if tx == 0:
        block_sums[cuda.blockIdx.x] = sdata[0]

N = 1_000_003
x = rng.random(N, dtype=np.float32)
nb = cdiv(N, 256)
block_sums = np.zeros(nb, np.float32)
reduction_sum_shared[nb, 256](x, block_sums)
np.testing.assert_allclose(np.sum(block_sums), np.sum(x), rtol=1e-4)
print("✅ Task 14 passed")

✅ Task 14 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [18]:
# ===== Task 15: reduction_argmax_shared =====
T15_BIG = 1 << 62

@cuda.jit
def reduction_argmax_shared(in_arr, block_max_vals, block_max_indices):
    sv = cuda.shared.array(256, dtype=float32)
    si = cuda.shared.array(256, dtype=int64)
    tx = cuda.threadIdx.x
    i = cuda.grid(1)
    if i < in_arr.size:
        sv[tx] = in_arr[i]
        si[tx] = i
    else:
        sv[tx] = -math.inf
        si[tx] = T15_BIG
    cuda.syncthreads()
    s = cuda.blockDim.x // 2
    while s > 0:
        if tx < s:
            ov = sv[tx + s]
            oi = si[tx + s]
            # при равенстве значений побеждает меньший индекс
            if ov > sv[tx] or (ov == sv[tx] and oi < si[tx]):
                sv[tx] = ov
                si[tx] = oi
        cuda.syncthreads()
        s >>= 1
    if tx == 0:
        block_max_vals[cuda.blockIdx.x] = sv[0]
        block_max_indices[cuda.blockIdx.x] = si[0]

N = 100_003
x = rng.integers(0, 1000, N).astype(np.float32)       # много ничьих
nb = cdiv(N, 256)
bv = np.zeros(nb, np.float32); bi = np.zeros(nb, np.int64)
reduction_argmax_shared[nb, 256](x, bv, bi)
best = np.argmax(bv)
np.testing.assert_equal(bv[best], np.max(x))
assert bi[best] == np.argmax(x), (bi[best], np.argmax(x))
print("✅ Task 15 passed")

✅ Task 15 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [ ]:
# ===== Task 16: constant_memory_filter =====
FIR_WEIGHTS = (rng.random(64, dtype=np.float32) / 64).astype(np.float32)

@cuda.jit
def constant_memory_filter(signal, out):
    w = cuda.const.array_like(FIR_WEIGHTS)       # constant memory
    i = cuda.grid(1)
    n = signal.size
    if i < n:
        half = (64 - 1) // 2
        acc = float32(0.0)
        for j in range(64):
            g = i + half - j
            if g >= 0 and g < n:
                acc += w[j] * signal[g]
        out[i] = acc

N = 200_001
sig = rng.random(N, dtype=np.float32)
out = np.zeros(N, np.float32)
constant_memory_filter[cdiv(N, 256), 256](sig, out)
np.testing.assert_allclose(out, np.convolve(sig, FIR_WEIGHTS, mode="same"), rtol=1e-4, atol=1e-4)
print("✅ Task 16 passed")

In [19]:
# ===== Task 17: benchmark_pinned_transfer =====
def benchmark_pinned_transfer(size_bytes, repeats=5):
    n = int(size_bytes)
    pageable = rng.integers(0, 255, n, dtype=np.uint8)
    pinned = cuda.pinned_array(n, dtype=np.uint8)
    pinned[:] = pageable
    d = cuda.device_array(n, dtype=np.uint8)
    stream = cuda.stream()

    def timed(fn):
        fn(); stream.synchronize()
        s, e = cuda.event(), cuda.event()
        s.record(stream)
        for _ in range(repeats):
            fn()
        e.record(stream); e.synchronize()
        ms = cuda.event_elapsed_time(s, e) / repeats
        return n / ms / 1e6                      # GB/s

    res = {}
    res["h2d_pageable"] = timed(lambda: d.copy_to_device(pageable, stream=stream))
    res["h2d_pinned"] = timed(lambda: d.copy_to_device(pinned, stream=stream))
    out_pageable = np.empty(n, np.uint8)
    out_pinned = cuda.pinned_array(n, dtype=np.uint8)
    res["d2h_pageable"] = timed(lambda: d.copy_to_host(out_pageable, stream=stream))
    res["d2h_pinned"] = timed(lambda: d.copy_to_host(out_pinned, stream=stream))
    stream.synchronize()
    res["integrity"] = bool(np.array_equal(out_pinned, pinned) and np.array_equal(out_pageable, pinned))
    return res

res = benchmark_pinned_transfer(128 * 1024 * 1024)
for k, v in res.items():
    print(k, v if k == "integrity" else f"{v:.2f} GB/s")
assert res["integrity"]
assert res["h2d_pinned"] > res["h2d_pageable"]
ratio = res["h2d_pinned"] / res["h2d_pageable"]
if ratio < 1.5:
    print(f"⚠️ pinned/pageable = {ratio:.2f}x (< 1.5x по ТЗ) — на Colab результат зависит от хоста, перезапусти ячейку")
print("✅ Task 17 passed")

h2d_pageable 4.42 GB/s
h2d_pinned 12.30 GB/s
d2h_pageable 4.42 GB/s
d2h_pinned 13.13 GB/s
integrity True
✅ Task 17 passed


# Module 3 — Warp Intrinsics & Atomics (Tasks 18–25)

In [20]:
# ===== Task 18: warp_reduce_sum_shfl =====
@cuda.jit
def warp_reduce_sum_shfl(in_arr, out_arr):
    warp_sums = cuda.shared.array(32, dtype=float32)
    tx = cuda.threadIdx.x
    i = cuda.grid(1)
    val = float32(0.0)
    if i < in_arr.size:
        val = in_arr[i]
    val += cuda.shfl_down_sync(0xFFFFFFFF, val, 16)
    val += cuda.shfl_down_sync(0xFFFFFFFF, val, 8)
    val += cuda.shfl_down_sync(0xFFFFFFFF, val, 4)
    val += cuda.shfl_down_sync(0xFFFFFFFF, val, 2)
    val += cuda.shfl_down_sync(0xFFFFFFFF, val, 1)
    lane = tx % 32
    warp = tx // 32
    if lane == 0:
        warp_sums[warp] = val
    cuda.syncthreads()
    if warp == 0:                                   # первый варп агрегирует суммы варпов
        nwarps = cuda.blockDim.x // 32
        v = float32(0.0)
        if lane < nwarps:
            v = warp_sums[lane]
        v += cuda.shfl_down_sync(0xFFFFFFFF, v, 16)
        v += cuda.shfl_down_sync(0xFFFFFFFF, v, 8)
        v += cuda.shfl_down_sync(0xFFFFFFFF, v, 4)
        v += cuda.shfl_down_sync(0xFFFFFFFF, v, 2)
        v += cuda.shfl_down_sync(0xFFFFFFFF, v, 1)
        if lane == 0:
            out_arr[cuda.blockIdx.x] = v

N = 2**20 + 77
x = rng.random(N, dtype=np.float32)
nb = cdiv(N, 256)
out = np.zeros(nb, np.float32)
warp_reduce_sum_shfl[nb, 256](x, out)
np.testing.assert_allclose(np.sum(out), np.sum(x), rtol=1e-4)
print("✅ Task 18 passed")

✅ Task 18 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [21]:
# ===== Task 19: warp_vote_analyzer =====
# mask_out: shape (n_warps, 3) uint32 -> [ballot, all_sync, any_sync]
@cuda.jit
def warp_vote_analyzer(data, mask_out):
    FULL = 0xFFFFFFFF
    i = cuda.grid(1)
    pred = False
    if i < data.size:
        pred = data[i] > 0
    b = cuda.ballot_sync(FULL, pred)
    a = cuda.all_sync(FULL, pred)
    n = cuda.any_sync(FULL, pred)
    if cuda.threadIdx.x % 32 == 0:
        w = i // 32
        mask_out[w, 0] = b
        mask_out[w, 1] = 1 if a else 0
        mask_out[w, 2] = 1 if n else 0

n_warps = 64
data = np.zeros(32 * n_warps, np.float32)
data[0:32] = 1.0                                   # warp 0: все true
data[32:64] = -1.0                                 # warp 1: все false
data[64:96] = np.where(np.arange(32) % 2 == 0, 1.0, -1.0)   # warp 2: смешанный
data[96:] = rng.standard_normal(32 * n_warps - 96)
mask_out = np.zeros((n_warps, 3), np.uint32)
warp_vote_analyzer[cdiv(data.size, 128), 128](data, mask_out)

pred = (data > 0).reshape(n_warps, 32).astype(np.uint64)
exp_ballot = (pred << np.arange(32, dtype=np.uint64)).sum(axis=1).astype(np.uint32)
np.testing.assert_equal(mask_out[:, 0], exp_ballot)
np.testing.assert_equal(mask_out[:, 1], (pred.sum(1) == 32).astype(np.uint32))
np.testing.assert_equal(mask_out[:, 2], (pred.sum(1) > 0).astype(np.uint32))
print("ballot warp0/1/2:", [hex(v) for v in mask_out[:3, 0]])
print("✅ Task 19 passed")

ballot warp0/1/2: ['0xffffffff', '0x0', '0x55555555']
✅ Task 19 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [22]:
# ===== Task 20: scan_hillis_steele (один блок, N <= 1024) =====
@cuda.jit
def scan_hillis_steele(x, out):
    buf = cuda.shared.array((2, 1024), dtype=float32)   # ping-pong
    tx = cuda.threadIdx.x
    n = x.size
    if tx < n:
        buf[0, tx] = x[tx]
    else:
        buf[0, tx] = 0.0
    cuda.syncthreads()
    pin = 0
    stride = 1
    while stride < cuda.blockDim.x:
        pout = 1 - pin
        if tx >= stride:
            buf[pout, tx] = buf[pin, tx] + buf[pin, tx - stride]
        else:
            buf[pout, tx] = buf[pin, tx]
        cuda.syncthreads()
        pin = pout
        stride *= 2
    if tx < n:
        out[tx] = buf[pin, tx]

for n in [1, 100, 777, 1024]:
    x = rng.integers(0, 10, n).astype(np.float32)
    out = np.zeros(n, np.float32)
    scan_hillis_steele[1, 1024](x, out)
    np.testing.assert_allclose(out, np.cumsum(x), atol=1e-4)
print("✅ Task 20 passed")

✅ Task 20 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [23]:
# ===== Task 21: scan_blelloch_work_efficient (один блок, N <= 2048) =====
@cuda.jit(device=True)
def _cf(i):                      # padding индекса против bank conflicts
    return i + (i >> 5)

@cuda.jit
def scan_blelloch_work_efficient(x, out):
    temp = cuda.shared.array(2112, dtype=float32)      # 2048 + 2048/32
    tx = cuda.threadIdx.x
    n = x.size
    N = 2 * cuda.blockDim.x
    a = 2 * tx
    b = 2 * tx + 1
    if a < n:
        temp[_cf(a)] = x[a]
    else:
        temp[_cf(a)] = 0.0
    if b < n:
        temp[_cf(b)] = x[b]
    else:
        temp[_cf(b)] = 0.0
    # Up-sweep
    offset = 1
    d = N >> 1
    while d > 0:
        cuda.syncthreads()
        if tx < d:
            ai = offset * (2 * tx + 1) - 1
            bi = offset * (2 * tx + 2) - 1
            temp[_cf(bi)] += temp[_cf(ai)]
        offset *= 2
        d >>= 1
    if tx == 0:
        temp[_cf(N - 1)] = 0.0                          # обнуляем корень
    # Down-sweep
    d = 1
    while d < N:
        offset >>= 1
        cuda.syncthreads()
        if tx < d:
            ai = offset * (2 * tx + 1) - 1
            bi = offset * (2 * tx + 2) - 1
            t = temp[_cf(ai)]
            temp[_cf(ai)] = temp[_cf(bi)]
            temp[_cf(bi)] += t
        d *= 2
    cuda.syncthreads()
    # exclusive -> inclusive
    if a < n:
        out[a] = temp[_cf(a)] + x[a]
    if b < n:
        out[b] = temp[_cf(b)] + x[b]

for n in [1, 100, 1500, 2048]:
    x = rng.integers(0, 10, n).astype(np.float32)
    out = np.zeros(n, np.float32)
    scan_blelloch_work_efficient[1, 1024](x, out)
    np.testing.assert_allclose(out, np.cumsum(x), atol=1e-4)
print("✅ Task 21 passed")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


✅ Task 21 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [24]:
# ===== Task 22: histogram_privatized =====
@cuda.jit
def histogram_privatized(data, bins_out, num_bins):
    s_bins = cuda.shared.array(256, dtype=int32)
    tx = cuda.threadIdx.x
    k = tx
    while k < num_bins:
        s_bins[k] = 0
        k += cuda.blockDim.x
    cuda.syncthreads()
    start = cuda.grid(1)
    stride = cuda.gridsize(1)
    for i in range(start, data.size, stride):
        cuda.atomic.add(s_bins, data[i], 1)
    cuda.syncthreads()
    k = tx
    while k < num_bins:
        cuda.atomic.add(bins_out, k, s_bins[k])
        k += cuda.blockDim.x

N = 5_000_000
data = rng.integers(0, 256, N, dtype=np.uint8)
bins = np.zeros(256, np.int32)
histogram_privatized[64, 256](data, bins, 256)
np.testing.assert_equal(bins, np.histogram(data, bins=256, range=(0, 256))[0])
print("✅ Task 22 passed")

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


✅ Task 22 passed


In [25]:
# ===== Task 23: dynamic_work_queue =====
@cuda.jit
def dynamic_work_queue(queue_in, queue_out, counter):
    i = cuda.grid(1)
    if i < queue_in.size:
        v = queue_in[i]
        if v % 7 == 0:                                   # критерий отбора
            pos = cuda.atomic.add(counter, 0, 1)        # одна атомарная операция на проходящий поток
            queue_out[pos] = v

N = 1_000_003
q_in = rng.integers(0, 10_000, N).astype(np.int32)
q_out = np.zeros(N, np.int32)
counter = np.zeros(1, np.int32)
dynamic_work_queue[cdiv(N, 256), 256](q_in, q_out, counter)
ref = q_in[q_in % 7 == 0]
assert counter[0] == ref.size
np.testing.assert_equal(np.sort(q_out[:counter[0]]), np.sort(ref))
print("✅ Task 23 passed")

✅ Task 23 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [26]:
# ===== Task 24: atomic_cas_float_add =====
# arr_i — int32-представление (view) того же float32-буфера: CAS работает над целыми.
from numba.cuda import libdevice

@cuda.jit(device=True)
def atomic_cas_float_add(arr_i, index, val):
    old = arr_i[index]
    done = False
    while not done:
        assumed = old
        new_f = libdevice.int_as_float(assumed) + val
        new_i = libdevice.float_as_int(new_f)
        old = cuda.atomic.cas(arr_i, index, assumed, new_i)
        if old == assumed:
            done = True
    return libdevice.int_as_float(old)

@cuda.jit
def _hammer(arr_i, index, val):
    atomic_cas_float_add(arr_i, index, val)

d_arr = cuda.to_device(np.zeros(4, np.float32))
d_arr_i = d_arr.view(np.int32)
_hammer[4, 256](d_arr_i, 2, np.float32(1.0))             # 1024 потоков на одном слоте
res = d_arr.copy_to_host()
np.testing.assert_allclose(res[2], 1024.0, rtol=1e-4)
assert res[0] == res[1] == res[3] == 0
print("✅ Task 24 passed")

✅ Task 24 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [27]:
# ===== Task 25: sparse_dot_product_atomic =====
@cuda.jit
def sparse_dot_product_atomic(indices, values, dense_vec, result):
    start = cuda.grid(1)
    stride = cuda.gridsize(1)
    for k in range(start, indices.size, stride):
        cuda.atomic.add(result, 0, values[k] * dense_vec[indices[k]])

n_dense, nnz = 100_000, 50_000
indices = rng.integers(0, n_dense, nnz).astype(np.int32)
values = rng.random(nnz, dtype=np.float32)
dense = rng.random(n_dense, dtype=np.float32)
result = np.zeros(1, np.float32)
sparse_dot_product_atomic[64, 256](indices, values, dense, result)
np.testing.assert_allclose(result[0], np.dot(dense[indices], values), rtol=1e-4)
print("✅ Task 25 passed")

✅ Task 25 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


# Module 4 — Scientific Computing (Tasks 26–34)

In [28]:
# ===== Task 26: heat_diffusion_2d_step =====
@cuda.jit
def heat_diffusion_2d_step(u_curr, u_next, alpha, dt, dx, dy):
    j, i = cuda.grid(2)                      # j — x (колонка), i — y (строка)
    M, N = u_curr.shape
    if i < M and j < N:
        if i > 0 and i < M - 1 and j > 0 and j < N - 1:
            c = u_curr[i, j]
            lap = (u_curr[i, j + 1] - 2.0 * c + u_curr[i, j - 1]) / (dx * dx) \
                + (u_curr[i + 1, j] - 2.0 * c + u_curr[i - 1, j]) / (dy * dy)
            u_next[i, j] = c + alpha * dt * lap
        else:
            u_next[i, j] = u_curr[i, j]       # Dirichlet: края фиксированы

def cpu_jacobi_reference(u, alpha, dt, dx, dy):
    un = u.copy()
    lap = (u[1:-1, 2:] - 2*u[1:-1, 1:-1] + u[1:-1, :-2]) / (dx*dx) \
        + (u[2:, 1:-1] - 2*u[1:-1, 1:-1] + u[:-2, 1:-1]) / (dy*dy)
    un[1:-1, 1:-1] = u[1:-1, 1:-1] + alpha * dt * lap
    return un

M, N = 512, 700
u = rng.random((M, N), dtype=np.float32)
un = np.zeros_like(u)
alpha, dt, dx, dy = 0.1, 1e-3, 0.1, 0.1
heat_diffusion_2d_step[(cdiv(N, 16), cdiv(M, 16)), (16, 16)](u, un, alpha, dt, dx, dy)
np.testing.assert_allclose(un[1:-1, 1:-1], cpu_jacobi_reference(u, alpha, dt, dx, dy)[1:-1, 1:-1], atol=1e-5)
print("✅ Task 26 passed")

✅ Task 26 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [29]:
# ===== Task 27: wave_equation_1d_step =====
@cuda.jit
def wave_equation_1d_step(u_prev, u_curr, u_next, c, dt, dx):
    tile = cuda.shared.array(256 + 2, dtype=float32)     # radius-1 halo
    tx = cuda.threadIdx.x
    n = u_curr.size
    base = cuda.blockIdx.x * 256
    i = base + tx
    if i < n:
        tile[tx + 1] = u_curr[i]
    if tx == 0 and base > 0:
        tile[0] = u_curr[base - 1]
    if tx == 255 and base + 256 < n:
        tile[257] = u_curr[base + 256]
    cuda.syncthreads()
    if i < n:
        if i > 0 and i < n - 1:
            C2 = (c * dt / dx) * (c * dt / dx)
            u_next[i] = 2.0 * tile[tx + 1] - u_prev[i] + C2 * (tile[tx + 2] - 2.0 * tile[tx + 1] + tile[tx])
        else:
            u_next[i] = 0.0                              # фиксированные границы

def cpu_wave_step(u_prev, u_curr, c, dt, dx):
    C2 = (c * dt / dx) ** 2
    nxt = np.zeros_like(u_curr)
    nxt[1:-1] = 2*u_curr[1:-1] - u_prev[1:-1] + C2 * (u_curr[2:] - 2*u_curr[1:-1] + u_curr[:-2])
    return nxt

N = 100_003
c, dx = 1.0, 0.01
dt = 0.5 * dx / c                               # CFL: c*dt/dx = 0.5 <= 1
assert c * dt / dx <= 1.0
u_prev = rng.random(N, dtype=np.float32); u_curr = rng.random(N, dtype=np.float32)
u_next = np.zeros(N, np.float32)
wave_equation_1d_step[cdiv(N, 256), 256](u_prev, u_curr, u_next, c, dt, dx)
np.testing.assert_allclose(u_next, cpu_wave_step(u_prev, u_curr, c, dt, dx), atol=1e-4)
print("✅ Task 27 passed")

✅ Task 27 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [31]:
# ===== Task 28: nbody_all_pairs =====
T28_TILE = 128

@cuda.jit
def nbody_all_pairs(positions, velocities, masses, dt, softening):
    sx = cuda.shared.array(T28_TILE, dtype=float32)
    sy = cuda.shared.array(T28_TILE, dtype=float32)
    sz = cuda.shared.array(T28_TILE, dtype=float32)
    sm = cuda.shared.array(T28_TILE, dtype=float32)
    tx = cuda.threadIdx.x
    i = cuda.blockIdx.x * T28_TILE + tx
    n = positions.shape[0]
    xi = float32(0.0); yi = float32(0.0); zi = float32(0.0)
    if i < n:
        xi = positions[i, 0]; yi = positions[i, 1]; zi = positions[i, 2]
    ax = float32(0.0); ay = float32(0.0); az = float32(0.0)
    eps2 = softening * softening
    for tile in range((n + 127) // 128):
        j = tile * T28_TILE + tx
        if j < n:
            sx[tx] = positions[j, 0]; sy[tx] = positions[j, 1]; sz[tx] = positions[j, 2]
            sm[tx] = masses[j]
        else:
            sx[tx] = 0.0; sy[tx] = 0.0; sz[tx] = 0.0; sm[tx] = 0.0
        cuda.syncthreads()
        for k in range(T28_TILE):
            dx_ = sx[k] - xi
            dy_ = sy[k] - yi
            dz_ = sz[k] - zi
            r2 = dx_ * dx_ + dy_ * dy_ + dz_ * dz_ + eps2
            inv = 1.0 / math.sqrt(r2)
            s = sm[k] * inv * inv * inv
            ax += dx_ * s; ay += dy_ * s; az += dz_ * s
        cuda.syncthreads()
    if i < n:
        velocities[i, 0] += ax * dt
        velocities[i, 1] += ay * dt
        velocities[i, 2] += az * dt

def cpu_nbody_reference(pos, vel, mass, dt, soft):
    p = pos.astype(np.float64); m = mass.astype(np.float64)
    d = p[None, :, :] - p[:, None, :]                  # d[i,j] = p_j - p_i
    r2 = (d ** 2).sum(-1) + soft ** 2
    a = (d * (m[None, :, None] * r2[..., None] ** -1.5)).sum(1)
    return vel + a * dt

N = 1000
pos = rng.standard_normal((N, 3)).astype(np.float32)
vel = rng.standard_normal((N, 3)).astype(np.float32)
mass = (rng.random(N, dtype=np.float32) + 0.5)
dt, soft = 0.01, 0.1
vel_ref = cpu_nbody_reference(pos, vel, mass, dt, soft)
vel_gpu = vel.copy()
nbody_all_pairs[cdiv(N, T28_TILE), T28_TILE](pos, vel_gpu, mass, dt, soft)
np.testing.assert_allclose(vel_gpu, vel_ref, atol=1e-3)
print("✅ Task 28 passed")

✅ Task 28 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [32]:
# ===== Task 29: gaussian_blur_2d_separable =====
@cuda.jit(device=True)
def _reflect(i, n):                       # scipy mode='reflect': d c b a | a b c d | d c b a
    if i < 0:
        i = -i - 1
    if i >= n:
        i = 2 * n - i - 1
    return i

@cuda.jit
def _blur_h(src, dst, k1d):
    tile = cuda.shared.array((8, 48), dtype=float32)     # 32 + 2*R_max(8)
    tx = cuda.threadIdx.x; ty = cuda.threadIdx.y
    H, W = src.shape
    R = k1d.size // 2
    x = cuda.blockIdx.x * 32 + tx
    y = cuda.blockIdx.y * 8 + ty
    if y < H:
        t = tx
        while t < 32 + 2 * R:
            gx = _reflect(cuda.blockIdx.x * 32 - R + t, W)
            tile[ty, t] = src[y, gx]
            t += 32
    cuda.syncthreads()
    if x < W and y < H:
        acc = float32(0.0)
        for kk in range(2 * R + 1):
            acc += tile[ty, tx + kk] * k1d[kk]
        dst[y, x] = acc

@cuda.jit
def _blur_v(src, dst, k1d):
    tile = cuda.shared.array((24, 32), dtype=float32)    # 8 + 2*R_max(8)
    tx = cuda.threadIdx.x; ty = cuda.threadIdx.y
    H, W = src.shape
    R = k1d.size // 2
    x = cuda.blockIdx.x * 32 + tx
    y = cuda.blockIdx.y * 8 + ty
    if x < W:
        t = ty
        while t < 8 + 2 * R:
            gy = _reflect(cuda.blockIdx.y * 8 - R + t, H)
            tile[t, tx] = src[gy, x]
            t += 8
    cuda.syncthreads()
    if x < W and y < H:
        acc = float32(0.0)
        for kk in range(2 * R + 1):
            acc += tile[ty + kk, tx] * k1d[kk]
        dst[y, x] = acc

def gaussian_blur_2d_separable(img_in, img_out, temp_buf, kernel_1d):
    H, W = img_in.shape
    grid, block = (cdiv(W, 32), cdiv(H, 8)), (32, 8)
    _blur_h[grid, block](img_in, temp_buf, kernel_1d)     # горизонтальный проход -> temp_buf
    _blur_v[grid, block](temp_buf, img_out, kernel_1d)    # вертикальный проход -> img_out

sigma = 2.0
R = int(4.0 * sigma + 0.5)                                # = 8, как у scipy (truncate=4)
xs = np.arange(-R, R + 1)
k1d = np.exp(-xs**2 / (2 * sigma**2)); k1d = (k1d / k1d.sum()).astype(np.float32)
img = rng.random((500, 770), dtype=np.float32)
d_in = cuda.to_device(img)
d_tmp = cuda.device_array_like(d_in)
d_out = cuda.device_array_like(d_in)
gaussian_blur_2d_separable(d_in, d_out, d_tmp, cuda.to_device(k1d))
np.testing.assert_allclose(d_out.copy_to_host(), scipy.ndimage.gaussian_filter(img, sigma=2.0), atol=1e-3)
print("✅ Task 29 passed")

✅ Task 29 passed


In [33]:
# ===== Task 30: sobel_edge_detector =====
@cuda.jit
def sobel_edge_detector(img_in, magnitude, orientation):
    tile = cuda.shared.array((18, 18), dtype=float32)    # 16x16 + apron 1
    tx = cuda.threadIdx.x; ty = cuda.threadIdx.y
    H, W = img_in.shape
    bx0 = cuda.blockIdx.x * 16
    by0 = cuda.blockIdx.y * 16
    k = ty * 16 + tx
    while k < 324:
        ly = k // 18
        lx = k % 18
        gy = min(max(by0 - 1 + ly, 0), H - 1)            # clamp (edge)
        gx = min(max(bx0 - 1 + lx, 0), W - 1)
        tile[ly, lx] = img_in[gy, gx]
        k += 256
    cuda.syncthreads()
    x = bx0 + tx
    y = by0 + ty
    if x < W and y < H:
        p00 = tile[ty, tx];     p01 = tile[ty, tx + 1];     p02 = tile[ty, tx + 2]
        p10 = tile[ty + 1, tx];                             p12 = tile[ty + 1, tx + 2]
        p20 = tile[ty + 2, tx]; p21 = tile[ty + 2, tx + 1]; p22 = tile[ty + 2, tx + 2]
        gxv = -p00 + p02 - 2.0 * p10 + 2.0 * p12 - p20 + p22
        gyv = -p00 - 2.0 * p01 - p02 + p20 + 2.0 * p21 + p22
        magnitude[y, x] = math.sqrt(gxv * gxv + gyv * gyv)
        orientation[y, x] = math.atan2(gyv, gxv)

def cpu_sobel(img):
    p = np.pad(img.astype(np.float64), 1, mode="edge")
    gx = -p[:-2, :-2] + p[:-2, 2:] - 2*p[1:-1, :-2] + 2*p[1:-1, 2:] - p[2:, :-2] + p[2:, 2:]
    gy = -p[:-2, :-2] - 2*p[:-2, 1:-1] - p[:-2, 2:] + p[2:, :-2] + 2*p[2:, 1:-1] + p[2:, 2:]
    return np.sqrt(gx**2 + gy**2), np.arctan2(gy, gx)

img = rng.random((301, 517), dtype=np.float32)
mag = np.zeros_like(img); ori = np.zeros_like(img)
sobel_edge_detector[(cdiv(517, 16), cdiv(301, 16)), (16, 16)](img, mag, ori)
mag_ref, ori_ref = cpu_sobel(img)
np.testing.assert_allclose(mag, mag_ref, atol=1e-3)
np.testing.assert_allclose(ori, ori_ref, atol=1e-3)
print("✅ Task 30 passed")

✅ Task 30 passed


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:1089: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))
